# WhatsApp Data Ingestion and Cleaning

This notebook reads the WhatsApp export file, cleans it by removing links and media omitted messages, and displays the original vs processed lines.

In [ ]:
import re
import os

# Define the file path
file_path = rf"C:\Users\guga\Desktop\KayaChatBot\data\wpp\WhatsApp Chat with Kaya 👀 - gugu.txt"

# Check if file exists
if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    print(f"File found: {file_path}")

def clean_text(text):
    # Remove <Media omitted> lines
    if "<Media omitted>" in text:
        return None
    
    # Remove <This message was edited> tag
    text = text.replace("<This message was edited>", "")

    # Remove links (http/https)
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # Remove mentions (WhatsApp specific format with hidden chars \u2068 and \u2069)
    text = re.sub(r'@\u2068.*?\u2069', '', text)
    
    # Remove emojis (Broad unicode range)
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
    
    return text

# Generate output file path
file_dir = os.path.dirname(file_path)
file_name = os.path.basename(file_path)
name, ext = os.path.splitext(file_name)
output_file_path = os.path.join(file_dir, f"{name}_processed{ext}")

# List to store results for demonstration
results = []
cleaned_lines = []

# Regex to identify the start of a new message
# Format: 5/7/25, 00:49 - Name: Message
date_pattern = re.compile(r'^\d{1,2}/\d{1,2}/\d{2,4}, \d{1,2}:\d{2} - ')

raw_messages = []
current_message = ""

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    
    for line in lines:
        # Check if line starts with a timestamp
        if date_pattern.match(line):
            # If we have a current message accumulating, save it
            if current_message:
                raw_messages.append(current_message)
            current_message = line.strip()
        else:
            # It's a continuation of the previous message
            # Add a space before appending to keep separation
            if current_message:
                current_message += " " + line.strip()
    
    # Append the last message
    if current_message:
        raw_messages.append(current_message)

# Process the aggregated messages
for msg in raw_messages:
    processed_line = clean_text(msg)
    
    # Only keep if there is content left after cleaning
    if processed_line and processed_line.strip():
        # Check if the message part is empty (e.g. "Date - Name: ")
        # This happens if the message was only emojis or mentions
        if not processed_line.strip().endswith(':'):
             cleaned_lines.append(processed_line + '\n')
             results.append({
                "original": msg,
                "processed": processed_line
             })
        else:
             # Content was removed completely
             results.append({
                "original": msg,
                "processed": "<REMOVED>"
             })

# Write processed lines to file
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.writelines(cleaned_lines)

print(f"Total raw messages found: {len(raw_messages)}")
print(f"Lines saved to file: {len(cleaned_lines)}")
print(f"Processed file saved at: {output_file_path}\n")

print("--- Sample of Processed Lines (Original vs Cleaned) ---")
count = 0
for entry in results:
    # Show lines that were changed or removed
    if entry['original'] != entry['processed']:
        print(f"Original : {entry['original']}")
        print(f"Processed: {entry['processed']}")
        print("-" * 40)
        count += 1
        if count >= 20:  # Show first 20 modifications
            break